# Tabular / time series — training template

Renders for every `(category, framework)` pair in this family:
`tabular_classification`, `tabular_regression`, `time_series_classification`, `time_series_forecasting`.

This is a **template**, not a guide. The pod renders it for one use case with
`notebook.render(category, framework, context)`: `{{ ... }}` placeholders are
substituted from the context, and cells carrying
`metadata.tracebloc.applies_to` are dropped when they do not apply to the pair
being rendered. See `README.md` in this directory for the contract.

**Nothing here calls `start()`.** Start is a button, and there is no Run All.

## This run

| | |
|---|---|
| Use case | {{ use_case }} |
| Dataset | `{{ dataset_id }}` |
| Category | `{{ category }}` |
| Framework | `{{ framework }}` |
| Edges | {{ edge_count }} |
| Records per edge | {{ records_per_edge }} |

In [ ]:
# PENDING A RELEASED SDK. Once the image carries an SDK release with
# environment login, the pod is already authenticated -- it reads its scoped,
# short-lived credential from the environment, so there is no email/password
# prompt here and no token in the notebook.
#
# Until then `User()` PROMPTS interactively, which is wrong for a pod. Note
# that merged is not enough: environment login is on the SDK's `develop`
# (pyproject 1.0.9) but ABSENT from the latest tag v1.0.7, which is what
# `pip install tracebloc` resolves -- so this cell is contingent on a RELEASE,
# not on the change landing. Verified 2026-09-09: v1.0.7 contains no
# `env_login` module and no `TRACEBLOC_TOKEN` path at all.
from tracebloc import User

user = User()

In [ ]:
# Filled in by the model picker. Change the path to point at your own file.
MODEL_PATH = "{{ model_path }}"

user.upload_model(MODEL_PATH)

In [ ]:
training = user.link_model_dataset("{{ dataset_id }}")

In [ ]:
# ======================================================================
# Settings — the complete plan for this run, as plain SDK calls.
# Edit a value, then press Start. Nothing here calls start().
# ======================================================================

# --- Experiment ----------------------------------------------------------
training.experiment_name("{{ experiment_name }}")


# --- Federation ----------------------------------------------------------
# cycles and epochs are set below, gated on framework: the single-pass
# frameworks (sklearn, lifelines, scikit_survival) force both to 1 and print a
# red banner if you set them, so this family cannot offer one value to every
# pair.
training.aggregation_strategy("fedavg")

# --- Optimization --------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#1-optimizer
training.optimizer("sgd")
training.learning_rate({"type": "constant", "value": 0.001})
training.seed(0)                              # 0 means no fixed seed

# --- Data ----------------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#dataset-parameters-optional
training.validation_split({{ validation_split }})
training.training_classes({{ training_classes }})

# --- Preprocessing -------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#preprocessing-tabular-%26-time-series
training.handle_missing_values(True)
training.imputation_strategy("median")
# MinMaxScaler for time series, StandardScaler for tabular.
training.scaler("{{ scaler }}")

# --- Callbacks -----------------------------------------------------------
# https://docs.tracebloc.io/join-use-case/hyperparameters#callbacks
training.terminate_on_nan_callback()
# training.early_stop_callback(monitor="val_loss", patience=3)
# training.model_checkpoint_callback(monitor="val_loss", save_best_only=True)
# training.reduce_lr_callback(monitor="val_loss", factor=0.1, patience=2, min_delta=1e-4)

# --- Custom loss ---------------------------------------------------------
# Optional. Point at a `loss.py` next to your model file; it is validated at
# LINK time, so a loss that returns a non-scalar, a NaN, or a tensor detached
# from the graph is refused there rather than wasting a training run.
# https://docs.tracebloc.io/join-use-case/hyperparameters#3-loss-function
#
# training.loss_function({"type": "custom", "value": "loss.py"})
#
# Or pick a standard one:
# Or a standard one. THE VALID VALUES DEPEND ON THE PAIR, and the SDK
# refuses anything outside its allowlist by poisoning the plan, so Start
# then blocks -- an earlier draft suggested "crossentropy" here and that is
# wrong for several pairs this template renders for:
#   time_series_forecasting  mse | l1
#   time_to_event_prediction coxph
#   any sklearn pair         mse | binarycrossentropy   (custom loss.py refused)
#   lifelines / scikit_survival   a custom loss.py is IGNORED, not applied
#   everything else          crossentropy | mse | l1
# training.loss_function({"type": "standard", "value": "<see above>"})
#
# Per-category suggestions via gated fragments are tracked separately;
# until then this cell names the allowlist rather than guessing for you.


In [ ]:
# --- Feature interaction -------------------------------------------------
# Only rendered when the dataset sets `allow_feature_modification`. Derive a
# new column, or restrict training to a subset of columns:
#
# training.feature_interaction({"feature1": "age", "feature2": "bmi", "method": "product"})
# training.feature_interaction({"feature_list": ["age", "bmi"], "method": "include"})
#
# training.get_features()   # list the columns this dataset exposes

In [ ]:
# --- Federation ----------------------------------------------------------
# cycles = federated rounds; epochs = local epochs per round
# Cheap passes over small feature tables; fifteen rounds still sits inside the twenty-pass ceiling.
# https://docs.tracebloc.io/join-use-case/hyperparameters#training-parameters
training.cycles(15)
training.epochs(1)
# Plain FedAvg is safe here because epochs is 1: there is no local drift to
# correct. Raising epochs above 1 means moving to a drift-correcting strategy
# (fedprox, fedadam, fedyogi, fedadagrad) in the same edit.

In [ ]:
# --- Federation ----------------------------------------------------------
# This framework trains in a single pass: the SDK WARNS and sets cycles and epochs to
# 1; it does not refuse, so a value set here would be silently dropped. One
# round over the local data, then aggregation.

In [ ]:
# --- Feature shape -------------------------------------------------------
# Must agree with the dataset's column count, or the link is refused.
training.feature_points({{ feature_points }})

In [ ]:
# Column encoding and feature normalization. Not part of the forecasting
# path, which reads only the imputation pair above.
training.encoding_strategy("label")
training.normalize_features(True)

In [ ]:
# --- Time series ---------------------------------------------------------
# Lookback window: how many past steps are fed in as input.
training.sequence_length({{ sequence_length }})

In [ ]:
# How many future steps to predict. Forecasting only.
training.forecast_horizon({{ forecast_horizon }})

In [ ]:
# Emit a channel flagging where values were missing. Only the
# time-series-classification engine path produces these.
training.missingness_indicators(False)

## Start

Press **Start**. It re-links the model and dataset, executes the settings cell
above, and then starts the experiment — in that order, because `start()` is
one-shot and resets the plan. Your remaining team budget is shown beside the
button.

To iterate: change a value above and press Start again.

Prefer to leave? *Download .ipynb* and *Copy as script* both give you the same
settings as plain SDK calls.